# LoRA Fine-tuning Detoxify Multilingual Model

This notebook fine-tunes Detoxify's multilingual XLM-R model using LoRA adapters on Google Colab.

- Loads the Detoxify base model (unitary/multilingual-toxic-xlm-roberta)
- Applies LoRA adapters via PEFT
- Supports BCE or CE loss (configurable)
- Trains per-language and saves LoRA adapters
- Runs prediction on test set
- Computes evaluation metrics

## 1. Install Dependencies

In [ ]:
!pip install transformers datasets peft torch pyyaml pandas accelerate scikit-learn -q

## 2. Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
)
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configuration

Modify `DATA_DIR` to point to your data folder.

Expected directory structure:
```
DATA_DIR/
  processed/
    es/
      train.csv
      val.csv
      test.csv
```

In [ ]:
# ============== Configuration ==============

# Data directory (contains 'processed' subdirectory)
DATA_DIR = "/content/drive/MyDrive/Notebooks/data"

# Languages to fine-tune
languages = ["es"]  # e.g., ["es", "it", "tr"]

# Column names in CSV files
text_col = "comment_text"
label_col = "toxic"

# Output directory for LoRA adapters and predictions
output_root = Path("/content/drive/MyDrive/Notebooks/output/runs/lora_detoxify")
pred_dir = Path("/content/drive/MyDrive/Notebooks/output/predictions")

# Training hyperparameters
batch_size = 128
num_epochs = 3
lr = 2e-5
max_length = 256
weight_decay = 0.01

# Loss function: "bce", "weighted_bce", or "focal"
# - "bce": Binary Cross-Entropy (no class weighting)
# - "weighted_bce": Binary Cross-Entropy with class weighting (pos_weight = N_neg / N_pos)
# - "focal": Focal Loss (focuses on hard examples, good for imbalanced data)
loss_type = "weighted_bce"

# Focal Loss parameters (only used if loss_type = "focal")
focal_alpha = 0.25  # Weight for positive class (or set to None for auto-compute)
focal_gamma = 2.0   # Focusing parameter (higher = more focus on hard examples)

# Prediction settings
run_tag = "lora"  # Prefix for prediction files

# ============================================

# Derived paths
processed_dir = Path(DATA_DIR) / "processed"
detoxify_model_name = "unitary/multilingual-toxic-xlm-roberta"

# Flatten languages list if nested
if languages and isinstance(languages[0], list):
    languages = languages[0]

print(f"Languages to process: {languages}")
print(f"Loss type: {loss_type.upper()}")
print(f"Epochs: {num_epochs}, Batch size: {batch_size}, LR: {lr}")

## 5. Helper Functions

In [ ]:
def compute_pos_weight(labels: np.ndarray) -> float:
    """
    Compute positive-class weight for WeightedBCETrainer.
    pos_weight = N_neg / N_pos
    """
    labels = labels.astype(int)
    n_pos = (labels == 1).sum()
    n_neg = (labels == 0).sum()

    if n_pos == 0:
        return 1.0

    return float(n_neg / max(1, n_pos))

## 6. Custom Trainers (BCE, Weighted BCE, Focal)

In [ ]:
class BCETrainer(Trainer):
    """
    Custom Trainer that uses BCEWithLogitsLoss (no class weighting).
    For num_labels=1 models.
    """
    def compute_loss(self, model, inputs, return_outputs: bool = False, **kwargs):
        labels = inputs["labels"]
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits.view(-1)

        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels.float())

        if return_outputs:
            return loss, outputs
        return loss


class WeightedBCETrainer(Trainer):
    """
    Custom Trainer that uses BCEWithLogitsLoss with class weighting.
    For num_labels=1 models.
    """
    def __init__(self, pos_weight: float = 1.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = torch.tensor([pos_weight], dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs: bool = False, **kwargs):
        labels = inputs["labels"]
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits.view(-1)

        pos_weight = self.pos_weight.to(logits.device)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels.float())

        if return_outputs:
            return loss, outputs
        return loss


class FocalLossTrainer(Trainer):
    """
    Custom Trainer that uses Focal Loss.
    FL(p) = -alpha * (1 - p)^gamma * log(p)
    
    Good for highly imbalanced datasets - focuses on hard examples.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs: bool = False, **kwargs):
        labels = inputs["labels"]
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits.view(-1)
        
        # Compute probabilities
        probs = torch.sigmoid(logits)
        targets = labels.float()
        
        # Focal Loss computation
        # For positive samples (y=1): p_t = p, alpha_t = alpha
        # For negative samples (y=0): p_t = 1-p, alpha_t = 1-alpha
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        # Focal weight: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        
        # Binary cross entropy (without reduction)
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        
        # Apply focal weight and alpha
        loss = alpha_t * focal_weight * bce
        loss = loss.mean()

        if return_outputs:
            return loss, outputs
        return loss

## 7. Data Pipeline

In [ ]:
def prepare_datasets_for_language(
    processed_dir: Path,
    lang: str,
    text_col: str,
    label_col: str,
    tokenizer,
    max_length: int,
) -> Dict[str, Any]:
    """
    For a given language load train/val CSVs and tokenize them into HF Datasets.
    """
    lang_dir = processed_dir / lang
    train_path = lang_dir / "train.csv"
    val_path = lang_dir / "val.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing train split: {train_path}")
    if not val_path.exists():
        raise FileNotFoundError(f"Missing val split: {val_path}")

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    if text_col not in train_df.columns or label_col not in train_df.columns:
        raise KeyError(
            f"Expected columns '{text_col}' and '{label_col}' in {train_path}.\n"
            f"Found columns: {list(train_df.columns)}"
        )

    train_df[label_col] = train_df[label_col].astype(int)
    val_df[label_col] = val_df[label_col].astype(int)

    train_ds = Dataset.from_pandas(
        train_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
    )
    val_ds = Dataset.from_pandas(
        val_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
    )

    def tokenize_fn(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = batch["label"]
        return enc

    train_ds = train_ds.map(tokenize_fn, batched=True)
    val_ds = val_ds.map(tokenize_fn, batched=True)

    cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in train_ds.column_names:
        cols.append("token_type_ids")
    train_ds.set_format(type="torch", columns=cols)
    val_ds.set_format(type="torch", columns=cols)

    return {
        "train": train_ds,
        "val": val_ds,
        "train_labels": train_df[label_col].values,
    }

## 8. Model + LoRA

In [ ]:
def build_lora_model_from_detoxify() -> AutoModelForSequenceClassification:
    """
    Load Detoxify's multilingual XLM-R checkpoint and wrap it with LoRA adapters.
    Always uses num_labels=1 for binary toxicity classification.
    """
    print(f"Loading Detoxify checkpoint: {detoxify_model_name}")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        detoxify_model_name,
        num_labels=1,
        problem_type="single_label_classification",
        ignore_mismatched_sizes=True,
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query", "key", "value"],
        lora_dropout=0.1,
        bias="none",
        task_type="SEQ_CLS",
    )

    print("Applying LoRA adapters...")
    lora_model = get_peft_model(base_model, lora_config)
    
    return lora_model


def load_lora_model(adapter_path: Path):
    """
    Load a fine-tuned LoRA model from saved adapter.
    """
    base_model = AutoModelForSequenceClassification.from_pretrained(
        detoxify_model_name,
        num_labels=1,
        problem_type="single_label_classification",
        ignore_mismatched_sizes=True,
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model

## 9. Main Training

In [ ]:
# Device selection
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"Processed data dir: {processed_dir}")
print(f"Languages: {languages}")
print(f"Output root: {output_root}")
print(f"Loss type: {loss_type.upper()}")
print(f"Max length: {max_length}, Batch size: {batch_size}, Epochs: {num_epochs}, LR: {lr}")

# Load tokenizer
print(f"\nLoading tokenizer from {detoxify_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(detoxify_model_name, use_fast=True)

# Train a separate LoRA adapter for each language
for lang in languages:
    print("\n" + "=" * 70)
    print(f"[{lang}] Preparing data...")
    
    try:
        datasets = prepare_datasets_for_language(
            processed_dir=processed_dir,
            lang=lang,
            text_col=text_col,
            label_col=label_col,
            tokenizer=tokenizer,
            max_length=max_length,
        )
    except FileNotFoundError as e:
        print(f"[{lang}] Error: {e}")
        print(f"[{lang}] Skipping...")
        continue
        
    train_ds = datasets["train"]
    val_ds = datasets["val"]
    train_labels = datasets["train_labels"]

    print(f"[{lang}] Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

    # Compute class weights based on loss type
    pos_weight = 1.0
    if loss_type == "weighted_bce":
        pos_weight = compute_pos_weight(train_labels)
        print(f"[{lang}] Using Weighted BCE loss with pos_weight: {pos_weight:.4f}")
    elif loss_type == "focal":
        # Auto-compute alpha if not specified
        if focal_alpha is None:
            # Use proportion of positive samples as alpha
            alpha = (train_labels == 1).sum() / len(train_labels)
        else:
            alpha = focal_alpha
        print(f"[{lang}] Using Focal Loss with alpha={alpha:.4f}, gamma={focal_gamma}")
    else:
        print(f"[{lang}] Using BCE loss (no class weighting)")

    print(f"[{lang}] Building LoRA model from Detoxify checkpoint...")
    model = build_lora_model_from_detoxify()
    model.to(device)
    print(f"[{lang}] Model device: {next(model.parameters()).device}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[{lang}] Trainable params: {trainable_params:,} / {total_params:,} "
          f"({100 * trainable_params / total_params:.2f}%)")

    lang_out_dir = output_root / lang
    lang_out_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(lang_out_dir),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    # Select trainer based on loss type
    if loss_type == "weighted_bce":
        trainer = WeightedBCETrainer(
            pos_weight=pos_weight,
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
        )
    elif loss_type == "focal":
        trainer = FocalLossTrainer(
            alpha=alpha,
            gamma=focal_gamma,
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
        )
    else:
        # BCE loss without class weighting
        trainer = BCETrainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
        )

    print(f"[{lang}] Starting training...")
    trainer.train()
    print(f"[{lang}] Training complete.")

    save_path = lang_out_dir / "lora_adapter"
    save_path.mkdir(parents=True, exist_ok=True)
    print(f"[{lang}] Saving LoRA adapter to {save_path}")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    # Save loss_type info for prediction
    with open(save_path / "loss_type.txt", "w") as f:
        f.write(loss_type)

    print(f"[{lang}] ✓ LoRA adapter saved successfully.")
    
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("All languages finished. LoRA adapters saved to:")
print(f"  {output_root}")

## 10. Prediction on Test Set

In [ ]:
def predict_on_test(
    model,
    tokenizer,
    test_df: pd.DataFrame,
    text_col: str,
    max_length: int,
    batch_size: int,
    device,
) -> np.ndarray:
    """
    Run prediction on test dataframe, return probabilities.
    Always uses sigmoid since num_labels=1.
    """
    model.eval()
    model.to(device)
    
    texts = test_df[text_col].tolist()
    all_probs = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits.view(-1)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
    
    return np.array(all_probs)


# Run prediction for each language
pred_dir.mkdir(parents=True, exist_ok=True)

for lang in languages:
    print("\n" + "=" * 70)
    print(f"[{lang}] Running prediction...")
    
    # Load test data
    test_path = processed_dir / lang / "test.csv"
    if not test_path.exists():
        print(f"[{lang}] Test file not found: {test_path}")
        print(f"[{lang}] Skipping prediction...")
        continue
    
    test_df = pd.read_csv(test_path)
    print(f"[{lang}] Test samples: {len(test_df)}")
    
    # Load model
    adapter_path = output_root / lang / "lora_adapter"
    if not adapter_path.exists():
        print(f"[{lang}] Adapter not found: {adapter_path}")
        print(f"[{lang}] Skipping prediction...")
        continue
    
    print(f"[{lang}] Loading LoRA adapter from {adapter_path}...")
    model = load_lora_model(adapter_path)
    
    # Run prediction
    y_prob = predict_on_test(
        model=model,
        tokenizer=tokenizer,
        test_df=test_df,
        text_col=text_col,
        max_length=max_length,
        batch_size=batch_size,
        device=device,
    )
    
    # Save predictions
    pred_df = test_df.copy()
    pred_df["y_prob"] = y_prob
    pred_df["y_pred"] = (y_prob >= 0.5).astype(int)
    if label_col in pred_df.columns:
        pred_df["y_true"] = pred_df[label_col].astype(int)
    
    pred_file = pred_dir / f"{run_tag}_{lang}.csv"
    pred_df.to_csv(pred_file, index=False)
    print(f"[{lang}] ✓ Predictions saved to {pred_file}")
    
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("Prediction complete!")

## 11. Evaluation Metrics

In [ ]:
def compute_language_metrics(
    language: str, y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray
) -> Dict:
    """
    Compute per-language metrics: ROC-AUC, F1, Precision, Recall, Accuracy, confusion matrix stats.
    """
    metrics: Dict[str, float] = {}
    n_samples = int(len(y_true))
    n_positive = int((y_true == 1).sum())
    n_negative = n_samples - n_positive

    metrics["language"] = language
    metrics["n_samples"] = n_samples
    metrics["positive_rate"] = n_positive / n_samples if n_samples > 0 else 0.0

    try:
        if n_positive > 0 and n_negative > 0:
            metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        else:
            metrics["roc_auc"] = np.nan
    except Exception as e:
        print(f"Warning: Could not calculate ROC-AUC for {language}: {e}")
        metrics["roc_auc"] = np.nan

    metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
    metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
    metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
    metrics["accuracy"] = accuracy_score(y_true, y_pred)

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        metrics["true_positives"] = int(tp)
        metrics["true_negatives"] = int(tn)
        metrics["false_positives"] = int(fp)
        metrics["false_negatives"] = int(fn)
        metrics["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        metrics["fpr"] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        metrics["fnr"] = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    except Exception as e:
        print(f"Warning: Could not calculate confusion matrix for {language}: {e}")

    return metrics


def add_aggregate_rows(
    all_metrics: List[Dict],
    all_true: Dict[str, np.ndarray],
    all_prob: Dict[str, np.ndarray],
    all_pred: Dict[str, np.ndarray],
) -> pd.DataFrame:
    df = pd.DataFrame(all_metrics)

    metric_cols = [
        "roc_auc", "f1", "precision", "recall", "accuracy",
        "positive_rate", "specificity", "fpr", "fnr",
    ]

    macro_row: Dict[str, float] = {"language": "macro"}
    for col in metric_cols:
        if col in df.columns:
            macro_row[col] = df[col].mean()
    macro_row["n_samples"] = df["n_samples"].sum()
    df_macro = pd.DataFrame([macro_row])

    all_y_true = np.concatenate([all_true[lang] for lang in all_true])
    all_y_prob = np.concatenate([all_prob[lang] for lang in all_prob])
    all_y_pred = np.concatenate([all_pred[lang] for lang in all_pred])
    n_samples = int(len(all_y_true))
    n_positive = int((all_y_true == 1).sum())
    n_negative = n_samples - n_positive

    micro_row: Dict[str, float] = {"language": "micro", "n_samples": n_samples}
    micro_row["positive_rate"] = n_positive / n_samples if n_samples > 0 else 0.0

    try:
        if n_positive > 0 and n_negative > 0:
            micro_row["roc_auc"] = roc_auc_score(all_y_true, all_y_prob)
        else:
            micro_row["roc_auc"] = np.nan
    except Exception:
        micro_row["roc_auc"] = np.nan

    micro_row["f1"] = f1_score(all_y_true, all_y_pred, zero_division=0)
    micro_row["precision"] = precision_score(all_y_true, all_y_pred, zero_division=0)
    micro_row["recall"] = recall_score(all_y_true, all_y_pred, zero_division=0)
    micro_row["accuracy"] = accuracy_score(all_y_true, all_y_pred)

    try:
        tn, fp, fn, tp = confusion_matrix(all_y_true, all_y_pred, labels=[0, 1]).ravel()
        micro_row["true_positives"] = int(tp)
        micro_row["true_negatives"] = int(tn)
        micro_row["false_positives"] = int(fp)
        micro_row["false_negatives"] = int(fn)
        micro_row["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        micro_row["fpr"] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        micro_row["fnr"] = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    except Exception:
        pass

    df_micro = pd.DataFrame([micro_row])
    df_out = pd.concat([df, df_macro, df_micro], axis=0, ignore_index=True)
    return df_out

In [ ]:
# Compute metrics for all languages
print("Computing evaluation metrics...")

all_metrics: List[Dict] = []
all_true: Dict[str, np.ndarray] = {}
all_prob: Dict[str, np.ndarray] = {}
all_pred: Dict[str, np.ndarray] = {}

for lang in languages:
    pred_file = pred_dir / f"{run_tag}_{lang}.csv"
    if not pred_file.exists():
        print(f"[{lang}] Prediction file not found: {pred_file}")
        continue
    
    df = pd.read_csv(pred_file)
    
    if "y_true" not in df.columns:
        print(f"[{lang}] No ground truth labels in prediction file, skipping metrics.")
        continue
    
    y_true = df["y_true"].to_numpy().astype(int)
    y_prob = df["y_prob"].to_numpy().astype(float)
    y_pred = df["y_pred"].to_numpy().astype(int)
    
    metrics = compute_language_metrics(lang, y_true, y_prob, y_pred)
    all_metrics.append(metrics)
    all_true[lang] = y_true
    all_prob[lang] = y_prob
    all_pred[lang] = y_pred
    
    print(f"[{lang}] ROC-AUC: {metrics['roc_auc']:.4f}, F1: {metrics['f1']:.4f}, "
          f"Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}")

if all_metrics:
    df_metrics = add_aggregate_rows(all_metrics, all_true, all_prob, all_pred)
    
    column_order = [
        "language", "n_samples", "positive_rate", "roc_auc",
        "f1", "precision", "recall", "accuracy",
        "true_positives", "true_negatives", "false_positives", "false_negatives",
        "specificity", "fpr", "fnr"
    ]
    column_order = [c for c in column_order if c in df_metrics.columns]
    df_metrics = df_metrics[column_order]
    
    metrics_file = pred_dir / f"{run_tag}_metrics.csv"
    df_metrics.to_csv(metrics_file, index=False)
    print(f"\nMetrics saved to {metrics_file}")
    
    print("\n" + "=" * 70)
    print("Final Metrics:")
    print(df_metrics.to_string(index=False))
else:
    print("No metrics to compute.")